# 02 — Etiquetas P2O y transiciones

Simulador = bucle online del repo (`ExtremePointOnlineSession` + `ValidatorMask` + `encode_option`).

En cada paso se guardan los 35 features de **todas** las candidatas legales y el índice que elige el maestro.

Maestro: `privileged_volume_ep` (mayor volumen del buffer + mejor EP). La etiqueta ya es legal → label_rate 1.0 si hay opciones.

Los 5 de `examples/` **no se colectan**. No se entrena ningún modelo en este notebook.

In [2]:
from pathlib import Path
import sys

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

HERE = Path.cwd().resolve()
if HERE.name == "notebooks":
    ML_ROOT = HERE.parent
else:
    ML_ROOT = HERE if (HERE / "src_ml").is_dir() else Path("/home/edmundo/packing-services/online_policy_ml")

REPO_ROOT = ML_ROOT.parent
for p in (ML_ROOT / "src_ml", REPO_ROOT / "src"):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

from bootstrap import setup
setup()

import json
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

print("ML_ROOT =", ML_ROOT)

ML_ROOT = /home/edmundo/packing-services/online_policy_ml


In [3]:
from config import REGIMES, SEED, TEACHER_NAME
from paths import (
    BED_JSON_NAME,
    HOLDOUT_DIR,
    ORDER_IDS_NAME,
    REPORTS_DIR,
    TRAIN_DIR,
    VAL_DIR,
    TEST_DIR,
    transitions_path,
)
from splits import load_orders
from collect import collect_dataset, collect_order_transitions, load_transitions, save_transitions
from problems import order_to_problem
from packing_services.online.features import FEATURE_DIM, FEATURE_NAMES, FEATURE_VERSION

split_dirs = {"train": TRAIN_DIR, "val": VAL_DIR, "test": TEST_DIR}
for name, path in split_dirs.items():
    assert (path / BED_JSON_NAME).is_file(), f"Falta fase 1: {path / BED_JSON_NAME}"

datasets = {name: load_orders(path / BED_JSON_NAME) for name, path in split_dirs.items()}
ids = {name: json.loads((path / ORDER_IDS_NAME).read_text(encoding="utf-8")) for name, path in split_dirs.items()}
holdout_ids = json.loads((HOLDOUT_DIR / ORDER_IDS_NAME).read_text(encoding="utf-8"))
for name, oids in ids.items():
    leak = set(oids) & set(holdout_ids)
    assert not leak, f"{name} contiene demo: {leak}"

print("FEATURE_VERSION", FEATURE_VERSION, "FEATURE_DIM", FEATURE_DIM)
print("maestro:", TEACHER_NAME, "semilla:", SEED)
print("pedidos:", {k: len(v) for k, v in ids.items()})
print("regímenes:", REGIMES)

FEATURE_VERSION 1 FEATURE_DIM 35
maestro: privileged_volume_ep semilla: 42
pedidos: {'train': 24, 'val': 8, 'test': 8}
regímenes: ((1, 1), (3, 2))


## Un paso visible (train, p=1 s=1)

In [5]:
from packing_services.algorithms._constructive import order_items
from packing_services.domain.enums import SortStrategy
from packing_services.online.budget import InformationBudget
from packing_services.online.features import encode_option
from packing_services.online.mask import ValidatorMask
from packing_services.online.params import support_threshold
from packing_services.online.session import ExtremePointOnlineSession
from packing_services.online.types import StepOption
from teacher import label_index

oid = ids["train"][0]
orders = datasets["train"]
problem = order_to_problem(orders, oid, lookahead_p=1, select_s=1)
constraints = problem.constraints
session = ExtremePointOnlineSession(problem.containers, selection="best_fit")
mask = ValidatorMask(problem, min_support_ratio=support_threshold({}, constraints.basic_stability))
remaining = order_items(list(problem.items), SortStrategy.INPUT_ORDER)
select_s, observe_p = InformationBudget(observe_p=1, select_s=1).window(len(remaining))
selectable = remaining[:select_s]
preview = remaining[:observe_p]
options = []
for bi, item in enumerate(selectable):
    for cand in session.candidates(item, constraints):
        if mask.allows(cand, item, session, constraints):
            options.append(StepOption(item=item, candidate=cand, buffer_index=bi))
y = label_index(options)
row = encode_option(options[y], session=session, preview=preview, remaining_count=len(remaining))
print("order_id", oid, "n_items", len(problem.items))
print("opciones legales paso 0:", len(options), "| etiqueta:", y)
print("ítem:", options[y].item.id, "vol", round(options[y].item.volume), "rank_key", options[y].candidate.rank_key[:2])
print("vector dim", len(row), "(debe ser 35)")
assert len(row) == FEATURE_DIM == 35
assert 0 <= y < len(options)

order_id 00100090 n_items 14
opciones legales paso 0: 6 | etiqueta: 0
ítem: 00104306#1 vol 71400000 rank_key (-536500.0, 0.0)
vector dim 35 (debe ser 35)


## Colecta train / val / test (p=1 s=1 y p=3 s=2)

In [7]:
reports = []
for lookahead_p, select_s in REGIMES:
    print(f"\n=== p={lookahead_p} s={select_s} ===")
    for split_name in ("train", "val", "test"):
        payload = collect_dataset(
            datasets[split_name],
            ids[split_name],
            lookahead_p=lookahead_p,
            select_s=select_s,
            teacher=TEACHER_NAME,
        )
        path = transitions_path(split_name, lookahead_p, select_s)
        save_transitions(payload, path)
        n_opt = [t["n_options"] for t in payload["transitions"]]
        packed = [s["n_packed"] for s in payload["per_order"]]
        row = {
            "split": split_name,
            "p": lookahead_p,
            "s": select_s,
            "n_orders": len(ids[split_name]),
            "n_transitions": payload["n_transitions"],
            "label_rate": payload["label_rate"],
            "n_options_mean": round(sum(n_opt) / max(len(n_opt), 1), 2),
            "packed_mean": round(sum(packed) / max(len(packed), 1), 2),
            "seconds": round(payload["seconds"], 2),
        }
        reports.append(row)
        print(f"  {split_name}: {row['n_transitions']} transiciones, label_rate={row['label_rate']:.2f}, {row['seconds']}s → {path.name}")

df = pd.DataFrame(reports)
display(df)
assert df["label_rate"].min() == 1.0
assert df["n_transitions"].min() > 0
print("\nOK — transiciones escritas")


=== p=1 s=1 ===
  train: 314 transiciones, label_rate=1.00, 1.25s → transitions_p1s1.pkl
  val: 109 transiciones, label_rate=1.00, 0.39s → transitions_p1s1.pkl
  test: 104 transiciones, label_rate=1.00, 0.33s → transitions_p1s1.pkl

=== p=3 s=2 ===
  train: 315 transiciones, label_rate=1.00, 2.42s → transitions_p3s2.pkl
  val: 112 transiciones, label_rate=1.00, 0.87s → transitions_p3s2.pkl
  test: 105 transiciones, label_rate=1.00, 0.77s → transitions_p3s2.pkl
   split  p  s  n_orders  ...  label_rate  n_options_mean  packed_mean  seconds
0  train  1  1        24  ...         1.0           12.85        13.08     1.25
1    val  1  1         8  ...         1.0           11.40        13.62     0.39
2   test  1  1         8  ...         1.0           11.45        13.00     0.33
3  train  3  2        24  ...         1.0           26.12        13.12     2.42
4    val  3  2         8  ...         1.0           24.72        14.00     0.87
5   test  3  2         8  ...         1.0           24

In [8]:
sample = load_transitions(transitions_path("train", 1, 1))
tr0 = sample["transitions"][0]
print("feature_version", sample["feature_version"], "dim", sample["feature_dim"])
print("names == FEATURE_NAMES", sample["feature_names"] == list(FEATURE_NAMES))
print("paso 0: n_options", tr0["n_options"], "label", tr0["label"], "feat0", len(tr0["features"][0]))
assert sample["feature_names"] == list(FEATURE_NAMES)
assert 0 <= tr0["label"] < tr0["n_options"]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, (p, s) in zip(axes, REGIMES):
    data = load_transitions(transitions_path("train", p, s))
    ax.hist([t["n_options"] for t in data["transitions"]], bins=20, color="#6a1b9a")
    ax.set_title(f"Opciones legales / paso (p={p} s={s})")
    ax.set_xlabel("N")
fig.tight_layout()
plt.show()

feature_version 1 dim 35
names == FEATURE_NAMES True
paso 0: n_options 6 label 0 feat0 35


02_etiquetas_p2o.ipynb:cell7:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Comparación puntual de maestros (1 pedido, p=3 s=2)

`receding_horizon_ep` packea el resto con privilegio y proyecta. Puede diferir del de máximo volumen.

In [10]:
oid = ids["train"][0]
problem = order_to_problem(datasets["train"], oid, lookahead_p=3, select_s=2)
tr_vol, st_vol = collect_order_transitions(problem, lookahead_p=3, select_s=2, teacher="privileged_volume_ep")
tr_rh, st_rh = collect_order_transitions(problem, lookahead_p=3, select_s=2, teacher="receding_horizon_ep")
n_cmp = min(len(tr_vol), len(tr_rh))
n_diff = sum(a["label"] != b["label"] for a, b in zip(tr_vol[:n_cmp], tr_rh[:n_cmp]))
print("pedido", oid)
print("pasos volume / receding:", len(tr_vol), len(tr_rh))
print("etiquetas distintas:", n_diff, f"({100 * n_diff / max(n_cmp, 1):.1f}%)")
print("packed volume / receding:", st_vol["n_packed"], st_rh["n_packed"])
print("segundos:", round(st_vol["seconds"], 2), round(st_rh["seconds"], 2))

pedido 00100090
pasos volume / receding: 13 11
etiquetas distintas: 8 (72.7%)
packed volume / receding: 13 11
segundos: 0.1 0.26


In [11]:
out = REPORTS_DIR / "02_transitions.json"
payload = {
    "fase": 2,
    "teacher": TEACHER_NAME,
    "seed": SEED,
    "feature_version": FEATURE_VERSION,
    "feature_dim": FEATURE_DIM,
    "table": reports,
    "holdout_producto_excluded": holdout_ids,
}
out.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print("reporte", out)

reporte /home/edmundo/packing-services/online_policy_ml/artifacts/reports/02_transitions.json


Siguiente fase (aún no): entrenar linear / MLP y exportar checkpoints.